# DDPG Power Control Project - Google Colab Runner

This notebook runs the full pipeline for **DDPG-based adaptive uplink power control**:
- environment setup
- dependency installation
- model training
- evaluation
- result visualization
- artifact export


## 1) Locate or Clone Project

Choose one mode:
- **Clone mode**: set `USE_GIT_CLONE = True` and provide your repository URL.
- **Uploaded mode**: keep `USE_GIT_CLONE = False` and upload/extract the repo to Colab (for example under `/content`).


In [ ]:
import os
import subprocess
from pathlib import Path

USE_GIT_CLONE = False
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"
REPO_DIR_NAME = "ddpg-power-control"

def find_project_root(base: Path) -> Path | None:
    for root, dirs, files in os.walk(base):
        if "train_ddpg.py" in files and "evaluation.py" in files:
            return Path(root)
    return None

if USE_GIT_CLONE:
    if "<your-username>" in REPO_URL or "<your-repo>" in REPO_URL:
        raise ValueError("Set REPO_URL to your actual GitHub repository URL.")
    target_dir = Path("/content") / REPO_DIR_NAME
    if not target_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(target_dir)], check=True)
    WORKDIR = target_dir
else:
    candidates = [Path("/content"), Path.cwd()]
    WORKDIR = None
    for base in candidates:
        found = find_project_root(base)
        if found is not None:
            WORKDIR = found
            break
    if WORKDIR is None:
        raise FileNotFoundError("Could not locate project root. Upload/extract project files in /content or enable clone mode.")

os.chdir(WORKDIR)
print(f"Using project directory: {WORKDIR}")
print("Files:", sorted([p.name for p in Path('.').iterdir() if p.is_file()])[:15])


## 2) Install Dependencies


In [ ]:
import sys
import subprocess
from pathlib import Path

req_file = Path("requirements.txt")
if req_file.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)], check=True)
else:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "matplotlib", "gymnasium", "shimmy",
        "stable-baselines3", "torch", "pandas"
    ], check=True)
print("Dependencies installed successfully.")


## 3) Verify Imports and Runtime


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

print(f"NumPy: {np.__version__}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
%matplotlib inline


## 4) Train DDPG

The following command runs a quick, Colab-friendly training configuration.
Increase `--timesteps` for higher-quality training.


In [ ]:
!python train_ddpg.py --mode quick --users 4 --timesteps 8000 --log-level INFO


## 5) Run Evaluation

This computes baseline and DDPG metrics, prints a comparison table, and generates figures.


In [ ]:
!python evaluation.py --users 4 --eval-steps 2000 --runs 2 --log-level INFO


## 6) Display Generated Plots


In [ ]:
from pathlib import Path
from IPython.display import Image, display

plot_files = [
    "training_reward_curve.png",
    "training_reward_vs_timesteps.png",
    "sum_rate_comparison.png",
    "power_usage_comparison.png",
    "fairness_comparison.png",
    "power_efficiency_comparison.png",
    "user_rate_cdf.png",
]

for name in plot_files:
    candidates = [Path("results") / name, Path(name)]
    existing = next((p for p in candidates if p.exists()), None)
    if existing is not None:
        print(f"Displaying: {existing}")
        display(Image(filename=str(existing)))
    else:
        print(f"Not found: {name}")


## 7) Save/Export Artifacts

This copies key outputs into an `artifacts/` folder for easy download or sharing.


In [ ]:
from pathlib import Path
import shutil

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)

artifact_files = [
    "trained_ddpg_model.zip",
    "ddpg_power_control_model.zip",
    "training_episode_rewards.npy",
    "results/evaluation_summary.csv",
    "results/training_reward_curve.png",
    "results/training_reward_vs_timesteps.png",
    "results/sum_rate_comparison.png",
    "results/power_usage_comparison.png",
    "results/fairness_comparison.png",
    "results/power_efficiency_comparison.png",
    "results/user_rate_cdf.png",
]

for rel_path in artifact_files:
    src = Path(rel_path)
    if src.exists():
        dst = artifact_dir / src.name
        shutil.copy2(src, dst)
        print(f"Saved: {dst}")

print(f"Artifacts directory: {artifact_dir.resolve()}")
